<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/report_5_MDanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REPORT: Trajectory analysis with MDAnalysis

First save a copy of this notebook in your Google Drive (`File -> Save a copy in Drive`).

This is a group report. Each group analyses an MD simulation of a protein with the tools learned in the training notebook (`t05_MDanalysis`) and then uses an experimental structure of the same protein in a complex to ask a specific biological question.

The report has two parts:

1. **General analysis** (sections 3.1–3.4): apply the analyses of the training notebook to your system. Copy and adapt the code from the training notebook; the empty code cells are for you.
2. **System-specific analysis** (section 3.5): the code to set up the analysis is provided; you have to run it, complete it where indicated and interpret the results.

**Group:** 1 / 2

**Members:**

In [ ]:
from datetime import date

today = date.today()
print(today)

## The assignment

### GROUP 1: the angiotensin II type 1 receptor (AT1R)

AT1R is a class A G protein-coupled receptor (GPCR) and the main mediator of the effects of the peptide hormone angiotensin II on blood pressure. It is the target of the angiotensin receptor blockers (ARBs, the "sartans"), widely used against hypertension. Crystal structures are available for the inactive receptor bound to the antagonist ZD7155 ([4YAY](https://www.rcsb.org/structure/4YAY)) and to the inverse agonist olmesartan ([4ZUD](https://www.rcsb.org/structure/4ZUD)), and for the active receptor bound to angiotensin II and stabilised by a nanobody ([6OS0](https://www.rcsb.org/structure/6OS0)).

The simulations come from the [GPCRmd](https://www.gpcrmd.org) database ([entry 120](https://www.gpcrmd.org/dynadb/dynamics/id/120/)): the receptor bound to olmesartan, starting from the crystal structure 4ZUD and embedded in a lipid membrane, simulated in three independent replicas. The membrane, the water and the ions were removed from the files, which contain the receptor and olmesartan. Look at the GPCRmd page for the details of the simulations (force field, length, temperature) and cite the database in your report.

**Question:** does olmesartan keep its binding mode, which interactions hold it in the pocket, and does the receptor stay in its inactive conformation?

### GROUP 2: the C-terminal region of thrombospondin-1

Thrombospondins are extracellular glycoproteins that regulate the interactions between cells and the extracellular matrix. Their most conserved region is at the C-terminus: a series of calcium-binding type 3 (T3) repeats followed by a lectin-like C-terminal domain (CTD). The crystal structure of a fragment of human thrombospondin-1 ([1UX6](https://www.rcsb.org/structure/1UX6)) shows the last three T3 repeats wrapped around the CTD in a compact assembly. The T3 repeats have no secondary structure and are held together by a core of 12 calcium ions, while the CTD is a β-sandwich with four more calcium sites. An RGD motif in the T3 repeats, which binds cell-surface integrins, is more or less available depending on the calcium bound.

The simulations come from the [ATLAS](https://www.dsimb.inserm.fr/ATLAS) database ([entry 1ux6_A](https://www.dsimb.inserm.fr/ATLAS/database/ATLAS/1ux6_A/1ux6_A.html)): three replicas of 100 ns each with GROMACS and the CHARMM36m force field, in water with 150 mM NaCl at 300 K. Before the simulations, ATLAS removed from the crystal structure all the non-protein atoms, *including the calcium ions*: these are simulations of the protein without calcium. The water and the ions were removed from the files, which contain 1000 frames for each replica. ATLAS also renumbered the residues starting from 1: the file `1ux6_A_corresp.tsv` gives the correspondence with the numbering of 1UX6. Look at the ATLAS page for the details and cite the database in your report.

**Question:** what happens to a structure held together by calcium when the calcium is removed? Which parts of the protein lose their structure, and could this explain why calcium is essential for the function of thrombospondins?

# Part 1: setup

Run these cells at the start of every session.

In [ ]:
%pip install -q MDAnalysis py3Dmol biopython gdown

We work in a folder of the virtual machine (set `use_drive = True` to work in your Google Drive instead, for example in a folder shared by the group). The cell can be run more than once.

In [ ]:
import os
if 'BASE_DIR' not in globals():          # remember where we started: running this cell again does not go one folder deeper
    BASE_DIR = os.getcwd()

use_drive = False        # True: keep your files on Google Drive (Colab will ask your permission)
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    workdir = '/content/drive/MyDrive/Structural_Bioinformatics/T05'
else:
    workdir = os.path.join(BASE_DIR, 'T05')
os.makedirs(workdir, exist_ok=True)
%cd $workdir

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # hide some harmless warnings of MDAnalysis

import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
import py3Dmol

Choose your group and download the files of your simulation (only the first time):

In [ ]:
GROUP = 1      # 1 (AT1R) or 2 (thrombospondin-1)

import glob
import gdown

# the simulations are in a shared Google Drive folder, with one subfolder for each group
# containing a PDB file and one trajectory for each replica
SIMULATIONS = 'https://drive.google.com/drive/folders/1GCAogFhSKt8rBdkK4piy0GGd1bBxNiHk'
subfolder = {1: 'at1r', 2: '1ux6'}[GROUP]
folder = os.path.join('simulations', subfolder)

# list the files in the shared folder and download only those of your group (only the first time)
def find(pattern):
    return sorted(glob.glob(f'{folder}/**/{pattern}', recursive=True))

if not (find('*.pdb') and find('*.xtc') + find('*.dcd')):
    shared = gdown.download_folder(SIMULATIONS, output='simulations', skip_download=True, quiet=True)
    mine = [f for f in shared if f.path.replace(os.sep, '/').split('/')[0] == subfolder]
    assert mine, f'no subfolder {subfolder} in the shared folder'
    for f in mine:
        if not os.path.exists(f.local_path):
            os.makedirs(os.path.dirname(f.local_path), exist_ok=True)
            gdown.download(id=f.id, output=f.local_path, quiet=False)

pdb_files = find('*.pdb')
xtc_files = find('*.xtc') + find('*.dcd')
assert len(pdb_files) == 1, f'expected one PDB file in {folder}, found {pdb_files}: set pdb_file by hand'
pdb_file = pdb_files[0]
print('structure:   ', pdb_file)
print('trajectories:', xtc_files)

We load each trajectory in memory (so that it can be superimposed) and the PDB file alone as reference structure. `runs` is a list with one Universe for each replica and `u` is the first one.

In [ ]:
runs = [mda.Universe(pdb_file, xtc, in_memory=True) for xtc in xtc_files]
u = runs[0]
ref = mda.Universe(pdb_file)

# molecules that are not protein (the ligand), excluding caps of the termini, water and ions
LIGAND_SEL = 'not protein and not resname ACE NME NMA NH2 HOH WAT SOL TIP3 SOD CLA NA CL POT'

protein = u.select_atoms('protein')
print(f'{u.atoms.n_atoms} atoms, {protein.n_residues} protein residues '
      f'(numbered from {protein.residues.resids[0]} to {protein.residues.resids[-1]})')
print('other molecules:', sorted(set(u.select_atoms(LIGAND_SEL).resnames)) or 'none')
for xtc, run in zip(xtc_files, runs):
    print(f'{xtc}: {run.trajectory.n_frames} frames, one every {run.trajectory.dt:.0f} ps, '
          f'{run.trajectory.totaltime / 1000:.1f} ns')

# Part 2: look at the trajectories

Before any analysis, look at your simulations: many of the questions of the report start from something you can see. Use [VMD](https://www.ks.uiuc.edu/Research/vmd/) or [ChimeraX](https://www.cgl.ucsf.edu/chimerax/), both free for academic use, on your computer.

First get the files: download them from the [shared folder](https://drive.google.com/drive/folders/1GCAogFhSKt8rBdkK4piy0GGd1bBxNiHk) (the subfolder of your group), or run the next cell. On your own computer the files are already in the folder printed by the cell.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import files
    for name in [pdb_file] + xtc_files:
        files.download(name)          # the trajectories are large: this can take a few minutes
else:
    print('the files are in', os.path.abspath(folder))

### Opening the trajectories

**VMD.** Open the PDB file (`File -> New Molecule`, then *Browse* and *Load*), then, with the molecule selected in the main window, choose `File -> Load Data Into Molecule` and load a trajectory. From a terminal you can do both at once: `vmd at1r.pdb at1r_rep1.xtc` (use your file names). Group 1 can load each replica in a new molecule (each with the PDB file) to compare them.

**ChimeraX.** In the command line at the bottom of the window:

```
open at1r.pdb
open at1r_rep1.xtc structureModel #1
```

A slider to move along the trajectory should appear; if it does not, or if you close it, `coordset slider #1` shows it.

### Things to do

Choose at least three of the following and describe what you see in the report, with a figure. Come back to the trajectories when you interpret the analyses of Part 3: whenever a plot shows something interesting (a peak in the RMSF, a jump in the RMSD, a contact that breaks), go to that frame and look at it.

1. **Get an overview.** Show the protein as a cartoon coloured by secondary structure and the ligand (if any) as sticks, and play the trajectory.
    * VMD: `Graphics -> Representations`, *Drawing Method* `NewCartoon` and *Coloring Method* `Secondary Structure`; create a second representation with the selection `not protein and not water` and `Licorice`.
    * ChimeraX: `cartoon; show ligand; style ligand stick`, and play with the slider.

    *Which regions move the most? Does the structure stay folded?*

2. **Remove the rotation and translation.** The protein may rotate and move in the box, which makes its internal motions hard to see.
    * VMD: `Extensions -> Analysis -> RMSD Trajectory Tool`: select `protein and name CA`, click *Align* and then *RMSD* (this is the RMSD of section 3.1). In `Graphics -> Representations`, tab *Trajectory*, a *Trajectory Smoothing Window* of 5-10 frames makes the motions easier to follow.
    * ChimeraX: `coordset slider #1 holdSteady #1@CA` keeps the Cα atoms superimposed while you move along the trajectory.

3. **Look at the binding site.** Show the residues close to the ligand (Group 1) or the aspartates and glutamates of the T3 repeats, which bind calcium in the crystal structure (Group 2), and follow how they change.
    * VMD: Group 1, a representation with the selection `same residue as (protein within 4 of (not protein and not water))`, *Licorice*, and in the tab *Trajectory* tick *Update Selection Every Frame*; Group 2, the selection `resname ASP GLU and sidechain` with residue numbers of the T3 repeats (see section 3.5). `Extensions -> Analysis -> Hydrogen Bonds` counts the hydrogen bonds between two selections along the trajectory.
    * ChimeraX: Group 1, `contacts ligand reveal true` and `hbonds ligand reveal true` show the contacts and the hydrogen bonds in the current frame; Group 2, `show :ASP,GLU & sidechain` (add the residue range of the T3 repeats). Compare the first frame, a frame in the middle and the last one.

4. **Measure a distance along the trajectory.** For example the distance between the Cα atoms of residues 3.50 and 6.30 (Group 1, section 3.5) or between the side chains of two aspartates that bind the same calcium ion in 1UX6 (Group 2, section 3.5).
    * VMD: press `2` in the graphics window and click two atoms; then `Graphics -> Labels`, choose *Bonds*, select the label and open the tab *Graph* to plot it as a function of the frame (and save it).
    * ChimeraX: `distance :126@CA :235@CA` (adapt the residue numbers to your files): the distance is updated when you move along the trajectory.

5. **Follow the secondary structure.** VMD: `Extensions -> Analysis -> Timeline`, then `Calculate -> Calc. Sec. Struct.` shows the secondary structure of each residue in each frame. Compare it with the DSSP analysis of section 3.3.

6. **Make a figure or a movie for the report.**
    * VMD: `File -> Render` for images, `Extensions -> Visualization -> Movie Maker` for movies.
    * ChimeraX: `save figure.png supersample 3` for an image; for a movie: `movie record; coordset #1 holdSteady #1@CA; wait; movie encode trajectory.mp4`.

# Part 3: the report

Write the report in the text cells and add plots with the code cells. Remember that the times in MDAnalysis are in ps and the distances in Å.

## 1. Introduction

*Describe the protein, its function and the biological question. What is known from experiments? What do you expect to observe in the simulation, and what do you think cannot be observed on the time scale of the simulation?*

## 2. Methods

*Describe what you know about the simulations (system, number of replicas, length, saving interval, what was removed from the files) and the analyses you performed: which atoms you used for the superposition, which selections, which parameters (e.g. cutoffs, clustering threshold), which part of the trajectory.*

## 3. Results

### 3.1 Global properties: RMSD and radius of gyration

*Does the RMSD reach a plateau? Should the first part of the trajectory be excluded from the other analyses? Is the size of the protein stable? Group 1: do the three replicas behave in the same way? Analyse all of them (loop over `runs`) in this and in the following sections.*

Hint: the termini and long loops can dominate the RMSD. Compare the RMSD of all the Cα atoms with that of the residues in a helix or a strand in the first frame (you can find them with DSSP, section 3.3). For Group 1, the transmembrane helices are the natural choice for the superposition.

In [ ]:
# your code here

### 3.2 Local flexibility: RMSF

*Which regions are the most flexible? What are their structural features?*

In [ ]:
# your code here

### 3.3 Secondary structure and hydrogen bonds

*Is the secondary structure stable? Which hydrogen bonds are the most stable, and which structural elements do they hold together?*

In [ ]:
# your code here

### 3.4 Conformational space: pairwise RMSD, clustering and PCA

*Does the protein visit one or more groups of conformations? How many principal components describe most of the motion, and which parts of the protein move along the first ones? Look at the free-energy landscape and check the convergence comparing the two halves of the trajectory (Group 1: compare the three replicas instead, e.g. with the RMSIP between the PCs of each pair of replicas): how much can you trust your conclusions?*

In [ ]:
# your code here

### 3.5 System-specific analysis

To answer the biological question we need to know which residues of the simulated protein form the binding sites. We find them in an experimental structure: the residues close to the ligand (Group 1) or to the calcium ions (Group 2) form the binding sites. The residue numbering of the experimental structure may differ from that of the simulation (the crystallised protein may be a fusion with another protein, or the simulated protein may have been renumbered), so we match the residues by aligning the two sequences.

Run the next cell: it defines some functions used below.

In [ ]:
import urllib.request
from Bio.Align import PairwiseAligner, substitution_matrices
from MDAnalysis.lib.util import convert_aa_code
from MDAnalysis.analysis.dssp import DSSP


def fetch_pdb(pdb_id):
    """Download a structure from the PDB (if not already here) and load it in a Universe."""
    filename = f'{pdb_id}.pdb'
    if not os.path.exists(filename):
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', filename)
    return mda.Universe(filename)


def other_molecules(structure):
    """Residue names of the molecules that are not protein or water (ligands, ions, ...)."""
    others = structure.select_atoms('not protein and not resname HOH WAT SOL TIP3')
    return sorted(set(others.resnames))


def one_letter(resname):
    try:
        return convert_aa_code(resname)
    except ValueError:
        return 'X'


def sequence(residues):
    return ''.join(one_letter(r.resname) for r in residues)


def ca_residues(atomgroup):
    """The residues of an AtomGroup that have a CA atom."""
    return atomgroup.select_atoms('protein and name CA').residues


def align_residues(residues_1, residues_2):
    """Align the sequences of two lists of residues.
    Returns the aligned pairs (residue_1, residue_2) and the fraction of identical pairs."""
    aligner = PairwiseAligner()
    aligner.mode = 'local'             # extra residues at the ends (e.g. a fusion protein) are not aligned
    aligner.substitution_matrix = substitution_matrices.load('BLOSUM62')
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5
    alignment = aligner.align(sequence(residues_1), sequence(residues_2))[0]
    pairs = []
    for (start_1, end_1), (start_2, end_2) in zip(*alignment.aligned):
        pairs += [(residues_1[start_1 + k], residues_2[start_2 + k]) for k in range(end_1 - start_1)]
    identity = np.mean([one_letter(a.resname) == one_letter(b.resname) for a, b in pairs]) if pairs else 0
    return pairs, identity


def map_to_simulation(structure, simulation):
    """Find the protein chain of an experimental structure that corresponds to the simulated protein.
    Returns the chain identifier and a dictionary {residue of the structure: residue of the simulation}."""
    sim_residues = ca_residues(simulation.atoms)
    best = None
    for chain in sorted(set(structure.select_atoms('protein').chainIDs)):
        pairs, identity = align_residues(ca_residues(structure.select_atoms(f'chainID {chain}')), sim_residues)
        print(f'chain {chain}: {len(pairs)} residues aligned to the simulation, {identity:.0%} identical')
        if best is None or len(pairs) * identity > len(best[1]) * best[2]:
            best = (chain, pairs, identity)
    chain, pairs, identity = best
    print(f'-> chain {chain} corresponds to the simulated protein')
    return chain, {a.resindex: b for a, b in pairs}


def site_residues(structure, chain, partner, cutoff=4.5):
    """Residues of the chain with a heavy atom closer than cutoff (Å) to the partner selection."""
    return structure.select_atoms(f'protein and chainID {chain} and not name H* '
                                  f'and around {cutoff} ({partner})').residues


def to_simulation(residues, mapping, simulation):
    """The residues of the simulation corresponding to residues of the experimental structure."""
    missing = [r for r in residues if r.resindex not in mapping]
    if missing:
        print('not present in the simulation:', ' '.join(f'{r.resname}{r.resid}' for r in missing))
    return simulation.residues[[mapping[r.resindex].resindex for r in residues if r.resindex in mapping]]


def to_structure(residues, mapping, structure):
    """The residues of the experimental structure corresponding to residues of the simulation
    (all of them must be present in the structure)."""
    inverse = {sim.resindex: ri for ri, sim in mapping.items()}
    return structure.residues[[inverse[r.resindex] for r in residues]]


def present_in(residues, mapping):
    """The residues of the simulation that are present in the experimental structure."""
    mapped = {sim.resindex for sim in mapping.values()}
    return residues[[r.resindex in mapped for r in residues]]


def as_selection(residues):
    """A selection string for a group of residues of the simulation."""
    return 'resindex ' + ' '.join(str(i) for i in residues.resindices)


def names(residues):
    return ' '.join(f'{r.resname}{r.resid}' for r in residues)

We also need the RMSF of each residue (Cα atoms), for each replica. If you have already superimposed the trajectories in section 3.2, this does not change anything.

In [ ]:
from MDAnalysis.analysis import distances

ca = u.select_atoms('protein and name CA')
rmsf_runs = []
for run in runs:
    align.AlignTraj(run, ref, select='protein and name CA', in_memory=True).run()
    rmsf_runs.append(rms.RMSF(run.select_atoms('protein and name CA')).run().results.rmsf)
rmsf = np.mean(rmsf_runs, axis=0)      # average over the replicas


def run_times(run):
    """The time of each frame of a run, in ns."""
    return np.array([ts.time / 1000 for ts in run.trajectory])

times = run_times(u)


def plot_rmsf_sites(sites):
    """RMSF of all residues (each replica and the average) with the residues of each site highlighted.
    sites = {'name': residues}"""
    plt.figure(figsize=(12, 4))
    if len(runs) > 1:
        for i, values in enumerate(rmsf_runs):
            plt.plot(ca.resids, values, lw=0.5, alpha=0.6, label=f'replica {i + 1}')
    plt.plot(ca.resids, rmsf, color='gray', lw=1.5, label='all residues' if len(runs) == 1 else 'average')
    for site_name, residues in sites.items():
        is_site = np.isin(ca.resindices, residues.resindices)
        plt.plot(ca.resids[is_site], rmsf[is_site], 'o', label=site_name)
        print(f'{site_name}: average RMSF {rmsf[is_site].mean():.2f} Å')
    print(f'all residues: average RMSF {rmsf.mean():.2f} Å')
    plt.xlabel('Residue number')
    plt.ylabel('RMSF (Å)')
    plt.legend()
    plt.show()


def contact_frequency(runs, sel_1, sel_2, cutoff=4.5):
    """For each residue of sel_1, the fraction of the frames (of all the runs) in which any of its
    heavy atoms is closer than cutoff to a heavy atom of sel_2."""
    counts, n_frames = 0, 0
    for run in runs:
        group_1 = run.select_atoms(f'({sel_1}) and not name H*')
        group_2 = run.select_atoms(f'({sel_2}) and not name H*')
        residue_of_atom = np.searchsorted(group_1.residues.resindices, group_1.resindices)
        run_counts = np.zeros(group_1.n_residues)
        for ts in run.trajectory:
            pairs = distances.capped_distance(group_1.positions, group_2.positions, max_cutoff=cutoff,
                                              box=run.dimensions, return_distances=False)
            run_counts[np.unique(residue_of_atom[pairs[:, 0]])] += 1
        counts = counts + run_counts
        n_frames += run.trajectory.n_frames
    return runs[0].residues[group_1.residues.resindices], counts / n_frames   # residues of the first run


def min_distance(run, sel_1, sel_2):
    """The minimum distance between two selections in each frame of a run."""
    group_1, group_2 = run.select_atoms(sel_1), run.select_atoms(sel_2)
    return np.array([distances.distance_array(group_1.positions, group_2.positions, box=run.dimensions).min()
                     for ts in run.trajectory])

## GROUP 1: AT1R

#### The pocket of olmesartan

The ligand is in your files, so we can find the pocket directly from the simulations: the residues in contact with olmesartan (any heavy atoms closer than 4.5 Å) in most of the frames of the three replicas.

In [ ]:
ligand_sel = f'({LIGAND_SEL}) and not name H*'      # heavy atoms of the ligand
print('ligand:', sorted(set(u.select_atoms(ligand_sel).resnames)), u.select_atoms(ligand_sel).n_atoms, 'heavy atoms')

residues, frequency = contact_frequency(runs, 'protein', ligand_sel)
pocket = residues[frequency > 0.5]
print('in contact with the ligand for more than half of the simulation:', names(pocket))

show = frequency > 0.05
plt.figure(figsize=(12, 4))
plt.bar(range(show.sum()), frequency[show])
plt.xticks(range(show.sum()), [f'{r.resname}{r.resid}' for r in residues[show]], rotation=90)
plt.ylabel('Contact frequency with the ligand')
plt.show()

Let's compare with the crystal structure 4ZUD, where we define the pocket in the same way. 4ZUD is a fusion of AT1R with another protein (BRIL), used to help crystallisation, and its numbering may differ from that of the simulation: the residues are matched by aligning the sequences. In the output, check that the alignment found the right chain and that most residues are identical.

In [ ]:
inactive = fetch_pdb('4ZUD')
print('other molecules in 4ZUD:', other_molecules(inactive))
chain_inactive, map_inactive = map_to_simulation(inactive, u)

pocket_xtal = to_simulation(site_residues(inactive, chain_inactive, 'resname OLM'), map_inactive, u)
print('pocket in 4ZUD:                ', names(pocket_xtal))
print('only in the simulation:        ', names(pocket.difference(pocket_xtal)))
print('only in the crystal structure: ', names(pocket_xtal.difference(pocket)))

plot_rmsf_sites({'pocket': pocket})

*Which residues form the pocket in the simulations? Is it the same pocket as in the crystal structure? Is the pocket more or less flexible than the rest of the receptor?*

#### Does olmesartan keep its binding mode?

We superimpose the receptor (Cα atoms) on the starting structure and compute the RMSD of the ligand heavy atoms, without superimposing the ligand itself: this measures how much the ligand moves *inside the pocket*.

In [ ]:
plt.figure(figsize=(10, 4))
for i, run in enumerate(runs):
    r = rms.RMSD(run, ref, select='protein and name CA', groupselections=[ligand_sel]).run().results.rmsd
    plt.plot(r[:, 1] / 1000, r[:, 3], label=f'replica {i + 1}')
plt.xlabel('Time (ns)')
plt.ylabel('Ligand RMSD (Å)')
plt.legend()
plt.show()

In the crystal structure olmesartan is anchored mainly by Tyr35, Trp84 and Arg167 (in the second extracellular loop). We follow the minimum distance between the ligand and the side chain of each of them:

In [ ]:
anchors_xtal = inactive.select_atoms(f'chainID {chain_inactive} and protein and resid 35 84 167').residues
anchors = to_simulation(anchors_xtal, map_inactive, u)
print('anchors in the simulation:', names(anchors))
assert len(anchors) > 0, 'anchor residues not found: check the mapping above'

fig, axes = plt.subplots(1, len(anchors), figsize=(5 * len(anchors), 4), sharey=True)
for ax, residue in zip(np.atleast_1d(axes), anchors):
    side_chain = f'resindex {residue.resindex} and not backbone and not name H*'
    for i, run in enumerate(runs):
        ax.plot(run_times(run), min_distance(run, side_chain, ligand_sel), lw=0.8, label=f'replica {i + 1}')
    ax.set_title(f'{residue.resname}{residue.resid}')
    ax.set_xlabel('Time (ns)')
np.atleast_1d(axes)[0].set_ylabel('Minimum distance from the ligand (Å)')
np.atleast_1d(axes)[0].legend()
plt.show()

*Does olmesartan stay in its crystallographic pose in all the replicas? Which interactions are stable, and which break and form again? Are there interactions that are not in the crystal structure? Use the hydrogen bond analysis of the training notebook to find which atoms are involved.*

#### Is the receptor still inactive?

The hallmark of GPCR activation is an outward movement of the intracellular end of transmembrane helix 6 (TM6), which opens a cavity where the G protein binds. It is usually followed through the distance between TM3 and TM6 on the intracellular side.

To compare residues of different GPCRs, the Ballesteros-Weinstein scheme numbers each residue of a transmembrane helix as *helix.position*, where the most conserved residue of the helix is number 50. For example, the arginine of the conserved DRY motif at the end of TM3 is 3.50; in TM6, the conserved CWxP motif (SWxP in AT1R) contains Trp 6.48, so 6.30, at the intracellular end of TM6, is 18 residues before it. We find these residues by searching the motifs in the sequence of the simulation:

In [ ]:
import re

sim_residues = ca_residues(u.atoms)
sim_sequence = sequence(sim_residues)

for motif in ['DRY', '[CS]W.P', 'NP..Y']:
    for match in re.finditer(motif, sim_sequence):
        print(f'{motif}: {names(sim_residues[match.start():match.end()])}')

If the motifs are found only once, the next cell defines residues 3.50 and 6.30. Otherwise, set them by hand (`r350 = sim_residues[index]`). We compare the distance in the simulations with that in the inactive structure (4ZUD) and in an active structure, bound to angiotensin II and to a nanobody that mimics the G protein (6OS0).

In [ ]:
r350 = sim_residues[re.search('DRY', sim_sequence).start() + 1]      # R of DRY
w648 = re.search('[CS]W.P', sim_sequence).start() + 1                # W of CWxP (SWxP in AT1R)
r630 = sim_residues[w648 - 18]
print(f'3.50: {r350.resname}{r350.resid}   6.30: {r630.resname}{r630.resid}')

active = fetch_pdb('6OS0')
print('other molecules in 6OS0:', other_molecules(active))
chain_active, map_active = map_to_simulation(active, u)

def distance_in_structure(structure, mapping):
    """Distance between the CA atoms of the residues of the structure that correspond to 3.50 and 6.30."""
    a, b = to_structure(u.residues[[r350.resindex, r630.resindex]], mapping, structure).atoms.select_atoms('name CA')
    return np.linalg.norm(a.position - b.position)

plt.figure(figsize=(10, 4))
for i, run in enumerate(runs):
    plt.plot(run_times(run), min_distance(run, f'name CA and resindex {r350.resindex}',
                                          f'name CA and resindex {r630.resindex}'), lw=0.8, label=f'replica {i + 1}')
plt.axhline(distance_in_structure(inactive, map_inactive), color='tab:blue', ls='--', label='inactive, 4ZUD')
plt.axhline(distance_in_structure(active, map_active), color='tab:red', ls='--', label='active, 6OS0')
plt.xlabel('Time (ns)')
plt.ylabel('Cα distance 3.50 - 6.30 (Å)')
plt.legend()
plt.show()

*Does the receptor stay inactive? Is this what you expect for a receptor bound to an inverse agonist? How long does the activation of a GPCR take, and could you observe it in these simulations? Look at the PCA of section 3.4: does any of the first principal components involve TM6?*

**Try it:** the NPxxY motif at the end of TM7 is another switch that changes conformation upon activation. Follow the distance between the tyrosine of NPxxY (7.53) and residue 3.50 in the simulations and in the two structures.

In [ ]:
# your code here

## GROUP 2: thrombospondin-1

#### Domains and calcium sites in the crystal structure

In 1UX6 the T3 repeats go up to residue 936 and the CTD starts at residue 937 (numbering of 1UX6). We find the corresponding residues of the simulation, and the calcium ions of the crystal structure (residue name `CA`, not to be confused with the Cα atoms, also named `CA`).

In [ ]:
xtal = fetch_pdb('1UX6')
print('other molecules in 1UX6:', other_molecules(xtal))
chain_x, map_x = map_to_simulation(xtal, u)

T3_END = 936      # last residue of the T3 repeats, numbering of 1UX6
x_residues = ca_residues(xtal.select_atoms(f'protein and chainID {chain_x}'))
t3 = to_simulation(x_residues[x_residues.resids <= T3_END], map_x, u)
ctd = to_simulation(x_residues[x_residues.resids > T3_END], map_x, u)
print(f'T3 repeats: residues {t3.resids[0]}-{t3.resids[-1]} of the simulation; CTD: {ctd.resids[0]}-{ctd.resids[-1]}')

calcium = xtal.select_atoms('resname CA and not protein')
print(f'{calcium.n_atoms} calcium ions in 1UX6')

For each calcium ion, the protein atoms closer than 3 Å are its ligands (mostly the oxygen atoms of aspartate and glutamate side chains, and some backbone carbonyl oxygens). We find the same atoms in the simulation:

In [ ]:
sites = []      # for each calcium ion: domain, ligand atoms in 1UX6, ligand atoms in the simulation
for ion in calcium:
    ligands = xtal.select_atoms(f'protein and chainID {chain_x} and not name H* and around 3.0 index {ion.index}')
    sim_atoms = []
    for atom in ligands:
        if atom.residue.resindex in map_x:
            match = map_x[atom.residue.resindex].atoms.select_atoms(f'name {atom.name}')
            if len(match) == 1:
                sim_atoms.append(match[0].index)
    if len(sim_atoms) >= 2:
        domain = 'T3' if np.median(ligands.resids) <= T3_END else 'CTD'
        sites.append((domain, ligands, u.atoms[sim_atoms]))

for domain in ['T3', 'CTD']:
    print(f'{domain}: {sum(d == domain for d, _, _ in sites)} calcium sites')

ca_ligands = {domain: sum((sim.residues for d, _, sim in sites if d == domain), u.residues[[]]).unique
              for domain in ['T3', 'CTD']}
for domain, residues in ca_ligands.items():
    print(f'residues binding calcium in the {domain}:', names(residues))

#### Flexibility of the two domains

In [ ]:
plot_rmsf_sites({'T3 repeats': t3, 'CTD': ctd})
plot_rmsf_sites({'calcium ligands, T3': ca_ligands['T3'], 'calcium ligands, CTD': ca_ligands['CTD']})

*Which domain is more flexible? Are the residues that bind calcium more or less flexible than the others?*

We superimpose the CTD on the starting structure and follow the RMSD of the CTD and, without further superposition, of the T3 repeats: if the T3 repeats unwrap from the CTD, their RMSD increases while that of the CTD stays low. We also count the contacts between the two domains (pairs of heavy atoms closer than 4.5 Å).

In [ ]:
def count_contacts(run, sel_1, sel_2, cutoff=4.5):
    """Number of pairs of heavy atoms closer than cutoff in each frame."""
    group_1 = run.select_atoms(f'({sel_1}) and not name H*')
    group_2 = run.select_atoms(f'({sel_2}) and not name H*')
    return np.array([len(distances.capped_distance(group_1.positions, group_2.positions, max_cutoff=cutoff,
                                                   return_distances=False)) for ts in run.trajectory])

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i, run in enumerate(runs):
    r = rms.RMSD(run, ref, select=f'name CA and ({as_selection(ctd)})',
                 groupselections=[f'name CA and ({as_selection(t3)})']).run().results.rmsd
    axes[0].plot(r[:, 1] / 1000, r[:, 2], lw=0.8, label=f'replica {i + 1}')
    axes[1].plot(r[:, 1] / 1000, r[:, 3], lw=0.8, label=f'replica {i + 1}')
    axes[2].plot(run_times(run), count_contacts(run, as_selection(t3), as_selection(ctd)), lw=0.8, label=f'replica {i + 1}')
axes[0].set_ylabel('RMSD of the CTD (Å)')
axes[1].set_ylabel('RMSD of the T3 repeats (Å)')
axes[2].set_ylabel('Contacts T3 repeats - CTD')
for ax in axes:
    ax.set_xlabel('Time (ns)')
axes[0].legend()
plt.show()

*Does the fragment keep its compact assembly? Do the three replicas behave in the same way? Look at the trajectories to see what happens.*

#### What happens to the calcium sites?

Without calcium, the negatively charged side chains that surrounded each ion repel each other. For each site we measure its size as the average distance between its ligand atoms, and compare it with the size in the crystal structure, where the ion is present:

In [ ]:
from MDAnalysis.lib.distances import self_distance_array

def site_size(positions):
    return self_distance_array(positions).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, domain in zip(axes, ['T3', 'CTD']):
    domain_sites = [(x, sim) for d, x, sim in sites if d == domain]
    for i, run in enumerate(runs):
        change = []
        for ts in run.trajectory:
            change.append(np.mean([site_size(run.atoms[sim.indices].positions) - site_size(x.positions)
                                   for x, sim in domain_sites]))
        ax.plot(run_times(run), change, lw=0.8, label=f'replica {i + 1}')
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title(f'{domain}: {len(domain_sites)} sites')
    ax.set_xlabel('Time (ns)')
axes[0].set_ylabel('Site size - size in 1UX6 (Å)')
axes[0].legend()
plt.show()

*Do the calcium sites open? Is there a difference between the sites of the T3 repeats and those of the CTD? Remember that the simulation box contained Na⁺ ions, which are not in your files: could they partly replace calcium?*

#### The RGD motif

The RGD motif binds integrins on the cell surface, and its availability depends on the calcium bound to the T3 repeats. We count the protein atoms around its side chains, a simple measure of how buried it is:

In [ ]:
import re

sim_residues = ca_residues(u.atoms)
sim_sequence = sequence(sim_residues)
start = re.search('RGD', sim_sequence).start()
rgd = sim_residues[start:start + 3]
neighbours = sim_residues[max(0, start - 2):start + 5]     # RGD and the two residues on each side
print('RGD motif:', names(rgd))

plt.figure(figsize=(10, 4))
for i, run in enumerate(runs):
    plt.plot(run_times(run), count_contacts(run, f'({as_selection(rgd)}) and not backbone',
                                            f'protein and not ({as_selection(neighbours)})'), lw=0.8, label=f'replica {i + 1}')
plt.xlabel('Time (ns)')
plt.ylabel('Contacts of the RGD side chains')
plt.legend()
plt.show()

*Does the RGD motif become more exposed without calcium? What could this mean for cell adhesion?*

**Try it:**

1. The B-factors of 1UX6 (column `tempfactors` of the atoms in MDAnalysis) measure the fluctuations in the crystal, *with* calcium. Convert the B-factors of the Cα atoms into fluctuations, $\sqrt{3B/8\pi^2}$, and compare them with the RMSF of the simulation, *without* calcium. Which regions differ the most?
2. Compare the secondary structure of the CTD at the start and at the end of the simulations (DSSP, section 3.3): does the β-sandwich stay folded?

In [ ]:
# your code here

### 3.6 Visualisation

*Look at the trajectory and connect what you see with the analyses above. Use pictures in the report where they help.*

The next cell shows 50 frames of the first trajectory, with the ligand (if any) in green. The residues in `highlight` are shown as sticks: set it to one of the groups of residues of section 3.5.

In [ ]:
highlight = None      # e.g. pocket (Group 1), t3, ca_ligands['T3'] (Group 2)

step = max(1, u.trajectory.n_frames // 50)
u.atoms.write('movie.pdb', frames=u.trajectory[::step])

view = py3Dmol.view(width=700, height=500)
view.addModelsAsFrames(open('movie.pdb').read(), 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
if highlight is not None:
    view.addStyle({'resi': [int(r) for r in highlight.resids]}, {'stick': {}})
ligand_names = sorted(set(u.select_atoms(LIGAND_SEL).resnames))
if ligand_names:
    view.setStyle({'resn': ligand_names}, {'stick': {'colorscheme': 'greenCarbon'}})
view.zoomTo()
view.animate({'loop': 'forward', 'interval': 100})
view.show()

## 4. Discussion

*Interpret the results and answer the question of your group. Some points to consider:*

*   *Which of your observations are robust, and which could change with a longer simulation or with independent replicas? Use the convergence analysis of section 3.4.*
*   *What can, and what cannot, be learned from a simulation of this length about the function of the protein?*
*   *Which additional simulations would you run to answer the question better?*

## 5. Conclusions

## References

*   Michaud-Agrawal N, Denning EJ, Woolf TB, Beckstein O (2011) MDAnalysis: a toolkit for the analysis of molecular dynamics simulations. *J Comput Chem* 32:2319-2327
*   Ballesteros JA, Weinstein H (1995) Integrated methods for the construction of three-dimensional models and computational probing of structure-function relations in G protein-coupled receptors. *Methods Neurosci* 25:366-428
*   Zhang H et al. (2015) Structure of the angiotensin receptor revealed by serial femtosecond crystallography. *Cell* 161:833-844 (4YAY)
*   Zhang H et al. (2015) *J Biol Chem* (4ZUD)
*   Rodríguez-Espigares I et al. (2020) GPCRmd uncovers the dynamics of the 3D-GPCRome. *Nat Methods* 17:777-787
*   Wingler LM et al. (2020) Angiotensin and biased analogs induce structurally distinct active conformations within a GPCR. *Science* 367:888-892 (6OS0)
*   Kvansakul M, Adams JC, Hohenester E (2004) Structure of a thrombospondin C-terminal fragment reveals a novel calcium core in the type 3 repeats. *EMBO J* 23:1223-1233 (1UX6)
*   Vander Meersche Y, Cretin G, Gheeraert A, Gelly JC, Galochkina T (2024) ATLAS: protein flexibility description from atomistic molecular dynamics simulations. *Nucleic Acids Res* 52:D384-D392